# HyDE Demo：LlamaIndex

使用 LlamaIndex 的 `HyDEQueryTransform`：先生成假设答案，再使用假设答案查询索引。

In [ ]:
# 如果环境中还没有 LlamaIndex，可先执行：
# %pip install llama-index llama-index-llms-google-genai llama-index-embeddings-huggingface

import os
from dotenv import load_dotenv
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.indices.query.query_transform import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.google_genai import GoogleGenAI

load_dotenv()
Settings.llm = GoogleGenAI(
    model="gemini-3.1-flash-lite",
    api_key=os.environ["GEMINI_API_KEY"],
    temperature=0,
)
Settings.embed_model = HuggingFaceEmbedding(model_name="moka-ai/m3e-base")

In [ ]:
documents = SimpleDirectoryReader(
    input_dir="../knowledge_db/prompt_engineering",
    recursive=True,
).load_data()
index = VectorStoreIndex.from_documents(documents) # 创建向量索引
base_query_engine = index.as_query_engine(similarity_top_k=4)

# 创建一个 HyDE 查询转换器 用户问题 → LLM 生成假设答案 → 使用假设答案进行向量检索
# include_original=True 表示检索时同时保留原始问题。综合使用原始问题和假设答案，避免只依赖 LLM 生成的假设答案
hyde = HyDEQueryTransform(include_original=True) 
query_engine = TransformQueryEngine(
    base_query_engine,
    query_transform=hyde,
)

In [ ]:
question = "总结文本转换这篇文章的主要观点、方法和示例"
response = query_engine.query(question)
print(response)